# 🎥 VideoDB Multicam Quickstart

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/quickstart/Multicam_Quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

Welcome to the **VideoDB Multicam Quickstart**! Learn how to build professional multi-camera surveillance systems with real-time AI event detection and multi-angle composition.

### 🎯 What You'll Build

A complete AI-powered multicam surveillance system that demonstrates:

- **👁️ SEE**: Connect to 4 security cameras and preview live feeds
- **🧠 UNDERSTAND**: Index visual content across all cameras with AI analysis
- **🎬 ACT**: Detect events in real-time, receive WebSocket alerts, and create same-interval multi-angle videos

By the end, you'll have a working system that:
- Monitors 4 cameras simultaneously
- Detects specific events (people with trolley bags, crowd formation, unattended luggage)
- Sends real-time alerts via WebSocket
- Creates same-interval 2x2 grid videos for evidence review after camera clock alignment is confirmed

---

## 🛠 Setup & Installation

Let's start by installing the VideoDB Python SDK and connecting to your account.

---

### 📦 Install VideoDB

VideoDB is available as a [Python package](https://pypi.org/project/videodb). Run the cell below to install it.

In [1]:
!pip install -q videodb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.9/95.9 kB 7.4 MB/s eta 0:00:00


### 🔗 Connect to VideoDB

You'll need an API key to interact with VideoDB. Provide it securely when prompted below.

> 💡 **Tip:** Get your free API key from the [VideoDB Console](https://console.videodb.io). Get $20 in free API credits upon sign-up — no credit card required!

In [2]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
coll = conn.get_collection()

print("✅ Connected to VideoDB successfully!")

Please enter your VideoDB API Key: ··········
✅ Connected to VideoDB successfully!


---

## 👁️ SEE - Connect Multi-Camera System

First, we'll connect to **4 security cameras** monitoring a public plaza. VideoDB handles RTSP stream ingestion seamlessly.

---

### 📹 Configure Cameras

In [3]:
# Multi-camera configuration
CAMERA_CONFIG = {
    "cam1": {"name": "Plaza Overview", "url": "rtsp://samples.rts.videodb.io:8554/pub-cam1"},
    "cam2": {"name": "Main Walkway", "url": "rtsp://samples.rts.videodb.io:8554/pub-cam2"},
    "cam3": {"name": "Stairway Junction", "url": "rtsp://samples.rts.videodb.io:8554/pub-cam3"},
    "cam4": {"name": "Central Plaza", "url": "rtsp://samples.rts.videodb.io:8554/pub-cam4"},
}

print("📋 Camera Configuration:")
for cam_id, info in CAMERA_CONFIG.items():
    print(f"   {cam_id}: {info['name']}")

📋 Camera Configuration:
   cam1: Plaza Overview
   cam2: Main Walkway
   cam3: Stairway Junction
   cam4: Central Plaza


---
### Connect all streams

In [4]:
# Connect all cameras
print("🔌 Connecting to all cameras...\n")

streams = {}
for cam_id, cam_info in CAMERA_CONFIG.items():
    stream = coll.connect_rtstream(
        name=f"Surveillance_{cam_id}",
        url=cam_info["url"],
        store=True,  # enables recording storage
    )
    streams[cam_id] = {"stream": stream, "info": cam_info}
    print(f"✅ {cam_id}: {stream.id}")

print(f"\n✅ All {len(streams)} cameras connected!")

🔌 Connecting to all cameras...

✅ cam1: rts-019fa99a-2a17-77c1-8c6c-79f5c7386fd1
✅ cam2: rts-019fa99a-583c-70c0-80f2-9bd64267694a
✅ cam3: rts-019fa99a-60b9-73f0-bba4-ab7da898d7d4
✅ cam4: rts-019fa99a-6892-7441-8629-9aa267658a73

✅ All 4 cameras connected!


#### To reconnect to existing stream:

In [ ]:
# EXISTING_STREAM_IDS = {
#     "cam1": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam1 stream ID
#     "cam2": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam2 stream ID
#     "cam3": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam3 stream ID
#     "cam4": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam4 stream ID
# }

# print("🔌 Reconnecting to existing cameras...\n")

# streams = {}
# for cam_id, rtstream_id in EXISTING_STREAM_IDS.items():
#     stream = coll.get_rtstream(rtstream_id)
#     streams[cam_id] = {"stream": stream, "info": CAMERA_CONFIG[cam_id]}
#     print(f"✅ {cam_id}: {stream.id} ({CAMERA_CONFIG[cam_id]['name']})")

# print(f"\n✅ All {len(streams)} cameras reconnected!")

---

### 📺 Preview Live Feeds

Let's preview the last 2 minutes from each camera to verify connectivity:
> Wait for atleast 2 minuts before executing the following cell.

In [10]:
from IPython.display import HTML
import time

# Get timestamps for last 2 minutes
now = int(time.time())
ten_seconds_ago = now - 120

print("📹 Generating preview streams for all cameras...\n")

# Generate streams and collect stream URLs
stream_urls = {}
video_titles = []
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]

    # Must run both (critical for RTStream preview)
    player_url = stream.generate_stream(ten_seconds_ago, now)

    stream_url = stream.stream_url  # Get actual stream URL
    stream_urls[cam_id] = stream_url
    video_titles.append(cam_data['info']['name'])
    print(f"✅ {cam_data['info']['name']}")
    print(f"  {player_url}\n")

print("\n📺 Displaying all 4 cameras in 2x2 grid:\n")

# Create HTML with 4 videos in 2x2 grid
html_content = """
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; max-width: 720px;">
"""

# Populate HTML with iframes for each video
for i, (cam_id, url) in enumerate(stream_urls.items()):
    html_content += f'''
        <div style="text-align: center;">
            <h4>{video_titles[i]}</h4>
            <iframe src="{url}" width="100%" height="200" frameborder="0" allowfullscreen></iframe>
        </div>
    '''

html_content += '</div>'

# Display all 4 videos at once
display(HTML(html_content))

📹 Generating preview streams for all cameras...

✅ Plaza Overview
  https://player.videodb.io/watch?v=dmMQb7vICUM

✅ Main Walkway
  https://player.videodb.io/watch?v=fMCVUB76S0g

✅ Stairway Junction
  https://player.videodb.io/watch?v=LIHhKPYkfKI

✅ Central Plaza
  https://player.videodb.io/watch?v=Gs_IyT1qsVs


📺 Displaying all 4 cameras in 2x2 grid:



---

## 🧠 UNDERSTAND - AI-Powered Visual Analysis

Now we'll set up continuous AI-powered visual understanding and indexing across all 4 cameras. The AI will analyze frames every 10 seconds to detect people, objects, and unusual activity.

---

### 🔍 Configure Visual Understanding

In [7]:
# Continuous visual understanding configuration
VISUAL_UNDERSTANDING_CONFIG = {
    "segmentation": {
        "type": "time",
        "window": "10s"
    },
    "analyzer": {
        "type": "vlm",
        "name": "scene",
        "sampling": {"frame_count": 1},
        "config": {
            "prompt": """Analyze this surveillance footage. Describe:
            1. People with trolley bags, backpacks, or luggage
            2. Crowd behavior (groups gathering, movement patterns)
            3. Unusual objects (unattended items)
            Be specific about location and appearance."""
        }
    }
}

print("📋 Visual Understanding Configuration:")
print(f"   Analyzing every {VISUAL_UNDERSTANDING_CONFIG['segmentation']['window']}")
print(f"   AI Focus: People, luggage, crowd behavior")

📋 Visual Understanding Configuration:
   Analyzing every 10s
   AI Focus: People, luggage, crowd behavior


In [8]:
# Start one continuous visual understanding per camera
print("🔧 Creating visual understandings...\n")

scene_understandings = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]
    understanding = stream.understand(
        segmentation=VISUAL_UNDERSTANDING_CONFIG["segmentation"],
        analyzers=[VISUAL_UNDERSTANDING_CONFIG["analyzer"]],
        store=True,
    )
    scene_output = understanding.outputs.get("scene")
    scene_understandings[cam_id] = {
        "understanding": understanding,
        "output": scene_output,
    }
    print(f"✅ {cam_id}: understanding started")

print(f"\n✅ Continuous visual understanding active on all {len(scene_understandings)} cameras!")

🔧 Creating visual understandings...

✅ cam1: understanding started
✅ cam2: understanding started
✅ cam3: understanding started
✅ cam4: understanding started

✅ Continuous visual understanding active on all 4 cameras!


### 🗂️ Index the Continuous Outputs

Create a semantic index for each camera's scene output.

In [9]:
# Create one continuous scene index per camera
print("🔧 Creating visual indexes...\n")

scene_indexes = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]
    understanding = scene_understandings[cam_id]["understanding"]
    scene_output = scene_understandings[cam_id]["output"]
    scene_index = stream.index(
        source=scene_output,
        name=f"Surveillance_{cam_id}_Index",
        use_for=["semantic"],
    )
    scene_indexes[cam_id] = {
        "understanding": understanding,
        "output": scene_output,
        "index": scene_index,
        "index_id": scene_index.id,
    }
    print(f"✅ {cam_id}: {scene_index.id}")

print(f"\n✅ Continuous visual indexing active on all {len(scene_indexes)} cameras!")

🔧 Creating visual indexes...

✅ cam1: idx-0c97607d0dedac5b
✅ cam2: idx-5f05e9f5c411caee
✅ cam3: idx-71dd18a27eed7d4b
✅ cam4: idx-f5f2f74463b5ff24

✅ Continuous visual indexing active on all 4 cameras!


---

### 👀 Review Indexed Records

Let's inspect the continuous records produced for each camera:

In [11]:
import time
import json

for cam_id in streams.keys():
    scene_index = scene_indexes[cam_id]["index"]
    print(f"Reviewing 3 recent records for {cam_id} ({streams[cam_id]['info']['name']}):\n")

    crib_records_payload = scene_index.get_records(
        page=1,
        page_size=3,
    )

    # Process records to remove 'start' and 'end' fields
    processed_records_payload = crib_records_payload.copy()
    if "records" in processed_records_payload:
        new_records = []
        for record in processed_records_payload["records"]:
            new_record = record.copy()
            new_record.pop("start", None)
            new_record.pop("end", None)
            new_records.append(new_record)
        processed_records_payload["records"] = new_records

    print(json.dumps(processed_records_payload, indent=2, default=str))
    print("\n" + "-"*50 + "\n")

Reviewing 3 recent records for cam1 (Plaza Overview):

{
  "next_page": true,
  "records": [
    {
      "description": "Here\u2019s a structured observation of the footage:\n\n### 1) People with trolley bags, backpacks, or luggage\n- **Left side, near the low wall:** a person in a dark coat is standing beside a **black rolling suitcase/trolley bag**. Next to them, another person wearing a **bright yellow/orange backpack** is walking with a small group.\n- **Center-left foreground:** a person in a **green jacket** is walking with a person carrying a **large magenta/purple backpack**. The backpack is worn high on the back and is very visible.\n- **Mid-right area:** several people are carrying **backpacks** in dark colors (black/navy), especially one person in a blue jacket walking away from the camera.\n- **Far right edge:** a person with a **large black backpack** is moving along the sidewalk border.\n- **Near the center background:** a few individuals appear to have **small bags or ba

---

## 🎬 ACT - Event Detection & Multi-Angle Composition

Now comes the exciting part! We'll:
1. Create event detection rules
2. Monitor for alerts in real-time via WebSocket
3. Create same-interval multi-camera evidence videos after confirming the camera clocks are aligned

---

### 🚨 Define Events to Detect

In [12]:
# Define 3 surveillance events
EVENTS_CONFIG = [
    {"label": "person_with_trolley", "prompt": "Detect any person with a trolley bag or rolling suitcase."},
    {"label": "large_crowd_formation", "prompt": "Identify 5+ people gathering quickly."},
    {"label": "unattended_luggage", "prompt": "Detect luggage left unattended for 1+ minute."},
]

print("🎯 Creating event detection rules...\n")

events = {}
for cfg in EVENTS_CONFIG:
    event_id = conn.create_event(
        event_prompt=cfg["prompt"],
        label=cfg["label"]
    )
    events[cfg["label"]] = {"event_id": event_id}
    print(f"Event: {cfg['label']}")
    print(f"    ID: {event_id}")

print(f"\n✅ {len(events)} events ready for detection!")

🎯 Creating event detection rules...

Event: person_with_trolley
    ID: 45e52378464fbc6b
Event: large_crowd_formation
    ID: 47bb3d60011d8f16
Event: unattended_luggage
    ID: c91e3ed29606d866

✅ 3 events ready for detection!


---

### 🔌 Connect WebSocket for Real-Time Alerts

WebSockets let us receive alerts instantly as they happen:

In [13]:
import asyncio

# Connect to WebSocket
ws_wrapper = conn.connect_websocket()
ws = await ws_wrapper.connect()

print(f"✅ WebSocket connected!")
print(f"   Connection ID: {ws.connection_id}")

INFO:videodb.websocket_client:WebSocket connected with ID: gVfsOy8y1eO4KEjMDA==


✅ WebSocket connected!
   Connection ID: gVfsOy8y1eO4KEjMDA==


Create Alerts to recieve on the web0socket.

In [14]:
# Create alerts for all cameras × all events
print("🔔 Creating alerts...\n")

RTSTREAM_ALERT_CALLBACK_URL = "https://example.com"
alerts = {}
for cam_id, idx_data in scene_indexes.items():
    alerts[cam_id] = {}
    print(f"--- Alerts for Camera: {cam_id} ---\n")
    for label, evt in events.items():
        alert_id = idx_data["index"].create_alert(
            evt["event_id"],
            callback_url=RTSTREAM_ALERT_CALLBACK_URL,
            ws_connection_id=ws.connection_id
        )
        alerts[cam_id][label] = alert_id
        print(f"Alert: {label}")
        print(f"    ID: {alert_id}")
    print(f"\n")

total_alerts = len(alerts) * len(events)
print(f"\n✅ {total_alerts} alerts active (4 cameras × 3 events)")



🔔 Creating alerts...

--- Alerts for Camera: cam1 ---

Alert: person_with_trolley
    ID: 050b2188b40f74c9
Alert: large_crowd_formation
    ID: 061e6a4d9bf969e3
Alert: unattended_luggage
    ID: 9657046d8d45abec


--- Alerts for Camera: cam2 ---

Alert: person_with_trolley
    ID: 40805abcc44d7a11
Alert: large_crowd_formation
    ID: 84e23e1e6b971f49
Alert: unattended_luggage
    ID: 1346d37047136f8a


--- Alerts for Camera: cam3 ---

Alert: person_with_trolley
    ID: 52415d236bc74f2b
Alert: large_crowd_formation
    ID: 9a12478631b5e931
Alert: unattended_luggage
    ID: b1544c9436a3396e


--- Alerts for Camera: cam4 ---

Alert: person_with_trolley
    ID: ed7f396a1b292107
Alert: large_crowd_formation
    ID: c4cb2cf74ffefb00
Alert: unattended_luggage
    ID: 2fd33de0ea1548a9



✅ 12 alerts active (4 cameras × 3 events)


---

### 👂 Listen for Alerts

Let's listen for 30 seconds. When events are detected, we'll receive instant notifications:

In [15]:
import json

# Create mapping from rtstream_id to cam info for easy lookup
rtstream_to_cam = {
    streams[cam_id]["stream"].id: {"cam_id": cam_id, "cam_name": CAMERA_CONFIG[cam_id]["name"]}
    for cam_id in streams.keys()
}

# Store alerts for later analysis
received_alerts = []

def get_alert_data(alert):
    if not isinstance(alert, dict):
        return {}
    data = alert.get("data") or alert
    return data if isinstance(data, dict) else {}

async def listen_for_alerts():
    timeout = 30  # Adjustable
    print(f"Listening for alerts ({timeout} seconds)...")
    print("The basketball analysis system will trigger alerts when events are detected\n")

    try:
        async with asyncio.timeout(timeout):
            async for msg in ws.receive():
                if not isinstance(msg, dict):
                    print("WebSocket message:", repr(msg))
                    continue
                if msg.get("channel") == "alert":
                    received_alerts.append(msg)
                    print(f"\nALERT #{len(received_alerts)} RECEIVED!")
                    print(json.dumps(msg, indent=2, default=str))
    except asyncio.TimeoutError:
        print(f"\nListening complete! {len(received_alerts)} alert(s) received")

# Start listening
await listen_for_alerts()

Listening for alerts (30 seconds)...
The basketball analysis system will trigger alerts when events are detected


ALERT #1 RECEIVED!
{
  "channel": "alert",
  "timestamp": "2026-07-28T16:44:26.047338+00:00",
  "rtstream_id": "rts-019fa99a-2a17-77c1-8c6c-79f5c7386fd1",
  "rtstream_name": "Surveillance_cam1",
  "data": {
    "event_id": "alert-050b2188b40f74c9",
    "label": "person_with_trolley",
    "triggered": true,
    "confidence": 0.98,
    "start": 1785257048.2396746,
    "end": 1785257059.2497537,
    "player_url": "https://console.videodb.io/player?url=https://rt.stream.videodb.io/manifests/rts-019fa99a-2a17-77c1-8c6c-79f5c7386fd1/1785257048000000-1785257060000000.m3u8",
    "stream_url": "https://rt.stream.videodb.io/manifests/rts-019fa99a-2a17-77c1-8c6c-79f5c7386fd1/1785257048000000-1785257060000000.m3u8",
    "explanation": "A person in the left-center foreground is standing next to a small upright rolling suitcase, which matches the trolley bag/rolling suitcase alert conte

---

### 📊 Analyze Alert Metrics

In [16]:
# Count alerts by event type
by_type = {}
for alert in received_alerts:
    label = get_alert_data(alert).get("label", "unknown")
    by_type[label] = by_type.get(label, 0) + 1

print("📊 Alerts by Event Type:")
print("-" * 50)
for label, count in sorted(by_type.items(), key=lambda x: x[1], reverse=True):
    # Format label: person_with_trolley -> Person with Trolley
    formatted_label = label.replace("_", " ").title()
    print(f"   {formatted_label:.<35} {count}")

# Count alerts by camera (using rtstream_to_cam mapping)
by_cam = {}
for alert in received_alerts:
    rtstream_id = alert.get("rtstream_id", "unknown") if isinstance(alert, dict) else "unknown"
    cam_info = rtstream_to_cam.get(rtstream_id, {"cam_id": "unknown", "cam_name": "Unknown"})
    cam_key = f"{cam_info['cam_id']} - {cam_info['cam_name']}"
    by_cam[cam_key] = by_cam.get(cam_key, 0) + 1

print("\n📹 Alerts by Camera:")
print("-" * 50)
for cam, count in sorted(by_cam.items(), key=lambda x: x[1], reverse=True):
    print(f"   {cam:.<35} {count}")

print(f"\n✅ Total: {len(received_alerts)} alerts received")

📊 Alerts by Event Type:
--------------------------------------------------
   Person With Trolley................ 17
   Large Crowd Formation.............. 2
   Unattended Luggage................. 1

📹 Alerts by Camera:
--------------------------------------------------
   cam1 - Plaza Overview.............. 9
   cam3 - Stairway Junction........... 6
   cam4 - Central Plaza............... 5

✅ Total: 20 alerts received


---

### 🔍 Select Alert to Investigate

Let's investigate a "person with trolley" alert and create a multi-angle evidence video:

In [18]:
# Select the first available alert
selected = next(
    alert for alert in received_alerts
)
data = get_alert_data(selected)
start_time = data.get("start")
end_time = data.get("end")
print(f"🚨 Selected Alert: {data.get('label', 'N/A')}")
print(f"   Confidence: {data.get('confidence', 'N/A')}")
print(f"   Time: {selected.get('timestamp', 'N/A')}")

# Use videodb play_stream to play the selected alert's video
from videodb import play_stream
player_url = data.get("stream_url")
print(f"\n▶️ Playing stream for selected alert: {data.get('label', 'N/A')}")
play_stream(player_url)

🚨 Selected Alert: person_with_trolley
   Confidence: 0.98
   Time: 2026-07-28T16:44:26.047338+00:00

▶️ Playing stream for selected alert: person_with_trolley


---

### 🎥 Generate Same-Interval Multi-Camera Clips

We'll request the same Unix interval from all 4 cameras. Confirm that the source camera clocks are aligned before treating the clips as synchronized.

In [19]:
# Add 10 seconds of context around the alert
PADDING = 10
clip_start = int(start_time - PADDING)
clip_end = int(end_time + PADDING)
clip_duration = clip_end - clip_start

print(f"⏱️ Alert Time Window: {start_time} → {end_time}")
print(f"📹 With padding: {clip_start} → {clip_end}\n")

# Generate the same requested interval for all cameras
stream_urls = {}
player_urls_map = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]

    # Must run both (critical for RTStream)
    player_url = stream.generate_stream(clip_start, clip_end)
    stream_url = stream.stream_url

    stream_urls[cam_id] = stream_url
    player_urls_map[cam_id] = player_url

print(f"✅ Same requested interval generated for 4 cameras\n")
print("Verify camera clock alignment before treating these clips as synchronized.\n")

print("Player URLs for each camera:")
for cam_id, p_url in player_urls_map.items():
    cam_name = CAMERA_CONFIG[cam_id]["name"]
    print(f"{cam_id} : {cam_name}")
    print(f"    {p_url}")

⏱️ Alert Time Window: 1785257048.2396746 → 1785257059.2497537
📹 With padding: 1785257038 → 1785257069

✅ Same requested interval generated for 4 cameras

Verify camera clock alignment before treating these clips as synchronized.

Player URLs for each camera:
cam1 : Plaza Overview
    https://player.videodb.io/watch?v=L_QYfG3KGMU
cam2 : Main Walkway
    https://player.videodb.io/watch?v=8ZLbtmgyQZc
cam3 : Stairway Junction
    https://player.videodb.io/watch?v=op3HFWqtw00
cam4 : Central Plaza
    https://player.videodb.io/watch?v=EZgEof7sJ7Y


---

### 💾 Download Clips for Composition

Since the Timeline Editor requires VideoDB media IDs, we'll download these clips using ffmpeg.

In [20]:
import subprocess
import os

# Download all 4 clips using ffmpeg
print("📥 Downloading clips with ffmpeg...\n")

downloads = {}
for cam_id, stream_url in stream_urls.items():
    output_file = f"{cam_id}_clip.mp4"

    # Use ffmpeg to download HLS stream
    cmd = [
        'ffmpeg',
        '-y',  # Overwrite output file
        '-i', stream_url,  # Input HLS URL
        '-c', 'copy',  # Copy codec (no re-encoding)
        '-loglevel', 'error',  # Only show errors
        output_file
    ]

    print(f"   {cam_id}: Downloading...")
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        downloads[cam_id] = {"name": output_file}
        print(f"   ✅ {cam_id}: Saved as {output_file}\n")
    except subprocess.CalledProcessError as e:
        print(f"   ❌ {cam_id}: Failed - {e.stderr.decode()}")
        downloads[cam_id] = {"name": None, "error": str(e)}

print(f"✅ Downloaded {len([d for d in downloads.values() if d.get('name')])} clips!")

📥 Downloading clips with ffmpeg...

   cam1: Downloading...
   ✅ cam1: Saved as cam1_clip.mp4

   cam2: Downloading...
   ✅ cam2: Saved as cam2_clip.mp4

   cam3: Downloading...
   ✅ cam3: Saved as cam3_clip.mp4

   cam4: Downloading...
   ✅ cam4: Saved as cam4_clip.mp4

✅ Downloaded 4 clips!


#### 💾 Uploading back to VideoDB

In [21]:
# Upload clips back to VideoDB
print("📤 Uploading clips to VideoDB...\n")

videos = {}
for cam_id, dl_info in downloads.items():
    video = coll.upload(file_path=dl_info["name"])
    videos[cam_id] = {"video": video, "video_id": video.id}
    print(f"✅ {cam_id}: {video.id}\n")

print("✅ All clips uploaded and ready for composition!")

📤 Uploading clips to VideoDB...

✅ cam1: m-z-019fa99f-69d7-7542-b118-2e7bb7af8532

✅ cam2: m-z-019fa99f-c1e1-79a2-bf70-e5ff09a0cac0

✅ cam3: m-z-019fa99f-ed52-7b21-9d2c-bbfe3f6ad5df

✅ cam4: m-z-019fa9a0-1a82-7a20-ab84-b4ba96b2b623

✅ All clips uploaded and ready for composition!


---

### 🎬 Create 2x2 Multi-Camera Grid

Now compose the 4 camera angles requested from the same Unix interval into a 2x2 grid. The replay is time-aligned only when the source camera clocks are aligned. Think of it like layers in a video editor:

- **Timeline**: The canvas
- **Track**: One layer (each camera gets its own track)
- **Clip**: The video positioned in a specific quadrant
- **Asset**: The video media itself

In [22]:
from videodb.editor import (
    Timeline, Track, Clip, VideoAsset, Position, Offset, Fit,
    TextAsset, Font, Border, Shadow, Background, TextAlignment
)

# Get minimum duration
print("📏 Checking video durations...\n")
video_durations = {}
for cam_id, vid_info in videos.items():
    video = vid_info["video"]
    video_durations[cam_id] = video.length

final_duration = min(video_durations.values())

# Create timeline with medium gray background
timeline = Timeline(conn)
timeline.resolution = "1280x720"
timeline.background = "#404040"  # Gray for padding areas

# Define camera positions with offsets for symmetrical padding
cam_configs = [
    {"cam_id": "cam1", "position": Position.top_left, "offset": Offset(x=0.03, y=0.025)},
    {"cam_id": "cam2", "position": Position.top_right, "offset": Offset(x=-0.03, y=0.025)},
    {"cam_id": "cam3", "position": Position.bottom_left, "offset": Offset(x=0.03, y=-0.025)},
    {"cam_id": "cam4", "position": Position.bottom_right, "offset": Offset(x=-0.03, y=-0.025)},
]

print("🎬 Building multi-camera grid...\n")

# Add video tracks
for config in cam_configs:
    cam_id = config["cam_id"]
    clip = Clip(
        asset=VideoAsset(id=videos[cam_id]["video_id"]),
        duration=final_duration,
        fit=Fit.crop,
        position=config["position"],
        offset=config["offset"],
        scale=0.45,  # Larger videos with small padding
    )
    track = Track()
    track.add_clip(0, clip)
    timeline.add_track(track)

# Define label positions - positioned inside each video at top
label_configs = [
    {"cam_id": "cam1", "offset": Offset(x=-0.355, y=-0.45)},
    {"cam_id": "cam2", "offset": Offset(x=0.135, y=-0.45)},
    {"cam_id": "cam3", "offset": Offset(x=-0.355, y=0.05)},
    {"cam_id": "cam4", "offset": Offset(x=0.135, y=0.05)},
]

# Add camera labels
for config in label_configs:
    cam_id = config["cam_id"]
    cam_name = CAMERA_CONFIG[cam_id]["name"]

    label_text_content = f"{cam_id.upper()}: {cam_name}"

    label_text = TextAsset(
        text=label_text_content,
        font=Font(
            family="Clear Sans",
            size=24,
            color="#FFFFFF",
        ),
        background=Background(
          color="#000000",
          height=50,
          width=300,
          text_alignment=TextAlignment.center,
        ),
    )

    label_clip = Clip(
        asset=label_text,
        duration=final_duration,
        offset=config["offset"],
    )

    track = Track()
    track.add_clip(0, label_clip)
    timeline.add_track(track)

print(f"✅ Production-ready multi-camera grid complete!\n")
print(f"   Resolution: 1280x720")
print(f"   Duration: {final_duration:.2f}s")
print(f"   Layout: 2x2 grid with symmetrical padding")

📏 Checking video durations...

🎬 Building multi-camera grid...

✅ Production-ready multi-camera grid complete!

   Resolution: 1280x720
   Duration: 19.00s
   Layout: 2x2 grid with symmetrical padding


In [23]:
# Generate final multi-camera view
from videodb import play_stream

print("🎬 Generating final video...\n")

stream = timeline.generate_stream()

print("✅ Multi-camera grid ready!")
print(f"Stream: {stream}\n")

play_stream(stream)

🎬 Generating final video...

✅ Multi-camera grid ready!
Stream: https://play.videodb.io/v1/06cfe8a8-d258-440e-b127-1056eacc8afa.m3u8



<div style="background-color: #d4edda; color: #155724; padding: 12px; border-left: 5px solid #28a745; border-radius: 4px;">
    <strong>🎉 Success!</strong> You've created a professional same-interval multi-camera surveillance clip from 4 different angles.
</div>

---

## 🧹 Cleanup

When you're done, let's disconnect all resources:

In [24]:
print("🧹 Cleaning up resources...\n")

# Disable all alerts
for cam_id, cam_alerts in alerts.items():
    for label, alert_id in cam_alerts.items():
        scene_indexes[cam_id]["index"].disable_alert(alert_id)

print("✅ All alerts disabled")


# Stop all continuous indexes
for cam_id, idx_data in scene_indexes.items():
    idx_data["index"].stop()

print("✅ All continuous indexes stopped")


# Stop all source understandings
for cam_id, idx_data in scene_indexes.items():
    idx_data["understanding"].stop()

print("✅ All source understandings stopped")


# Stop all streams
for cam_id, cam_data in streams.items():
    cam_data["stream"].stop()

print("✅ All streams stopped")


# Close the WebSocket after every server-side resource is stopped
await ws_wrapper.close()
print("✅ WebSocket closed")
print("\n🎉 Cleanup complete!")

🧹 Cleaning up resources...

✅ All alerts disabled
✅ All continuous indexes stopped
✅ All source understandings stopped
✅ All streams stopped
✅ WebSocket closed

🎉 Cleanup complete!


---

# 🏁 Conclusion: Professional Multicam Surveillance

Congratulations! You've built a complete AI-powered multi-camera surveillance system.

### 🎯 What You Accomplished

**👁️ SEE:**
- ✅ Connected 4 RTSP camera streams
- ✅ Previewed live feeds from multiple angles

**🧠 UNDERSTAND:**
- ✅ Indexed visual content across all cameras with AI
- ✅ Reviewed raw continuous index records from different angles

**🎬 ACT:**
- ✅ Created 3 event detection rules
- ✅ Monitored real-time alerts via WebSocket
- ✅ Analyzed alert metrics by type and camera
- ✅ Generated same-interval multi-camera clips with an explicit clock-alignment check
- ✅ Composed a professional 2x2 grid video

---

### 🚀 What's Next?

Ready to build more advanced surveillance systems?

- 📖 **[VideoDB Documentation](https://docs.videodb.io)**: Complete API reference
- 🍳 **[VideoDB Cookbook](https://github.com/video-db/videodb-cookbook)**: More multicam examples
- 💬 **[Discord Community](https://discord.com/invite/py9P639jGz)**: Get help and share projects

---

**Build intelligent surveillance systems with VideoDB — the perception layer for real-world AI.** 🎥

